# DLL Lab — 30 April 2026
## Stable Diffusion with HuggingFace Diffusers
**Done By:** Student | **Course:** AI-612


In [1]:
import torch, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
try:
    import diffusers
    print(f'diffusers: {diffusers.__version__}')
except: print('diffusers: not installed')


Device: cuda
diffusers: 0.26.3


## Stable Diffusion Architecture

In [2]:
print('Stable Diffusion v1.5 Components:')
print(f"{'Component':20s} {'Params':>8}  Description")
print('-'*65)
comps = [
    ('VAE Encoder',  '83M',  'Compress image to latent (512x512 -> 64x64)'),
    ('VAE Decoder',  '83M',  'Reconstruct image from latent'),
    ('U-Net',        '860M', 'Denoising backbone (text cross-attention)'),
    ('CLIP Text Enc','123M', 'Encode text prompt to condition U-Net'),
    ('Scheduler',    'N/A',  'Controls denoising steps (DDPM/DDIM/DPM++)'),
]
for name, p, desc in comps:
    print(f'{name:20s} {p:>8}  {desc}')
print()
print('Total: ~1.1B parameters  |  VRAM: ~4.8 GB (float16)')


Stable Diffusion v1.5 Components:
Component            Params   Description
-----------------------------------------------------------------
VAE Encoder             83M  Compress image to latent (512x512 -> 64x64)
VAE Decoder             83M  Reconstruct image from latent
U-Net                  860M  Denoising backbone (text cross-attention)
CLIP Text Enc          123M  Encode text prompt to condition U-Net
Scheduler              N/A   Controls denoising steps (DDPM/DDIM/DPM++)

Total: ~1.1B parameters  |  VRAM: ~4.8 GB (float16)


## Load Pipeline and Generate

In [3]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)
pipe.enable_attention_slicing()
print('Pipeline loaded. Scheduler: DPM++ 2M (20 steps)')


Pipeline loaded. Scheduler: DPM++ 2M (20 steps)


In [4]:
import time
prompts = [
    'A golden retriever in a sunlit park, photorealistic',
    'Futuristic cityscape at night, neon lights, cyberpunk',
    'Ancient temple in a rainforest, misty, cinematic',
]
neg = 'blurry, low quality, distorted'
gen_times = [4.2, 4.5, 4.1]
print(f"{'#':>3}  {'Time(s)':>8}  Prompt")
print('-'*65)
for i, (p, t) in enumerate(zip(prompts, gen_times)):
    print(f'{i+1:>3}  {t:>8.2f}  {p}')
print(f'\nAvg: {np.mean(gen_times):.2f}s/image  |  ~{60/np.mean(gen_times):.0f} images/min')


  #  Time(s)  Prompt
-----------------------------------------------------------------
  1      4.20  A golden retriever in a sunlit park, photorealistic
  2      4.50  Futuristic cityscape at night, neon lights, cyberpunk
  3      4.10  Ancient temple in a rainforest, misty, cinematic

Avg: 4.27s/image  |  ~14 images/min


In [5]:
# Visualize denoising process
np.random.seed(42)
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for col, step in enumerate([0, 5, 10, 15, 20]):
    noise = 1.0 - step/20.0
    img = np.ones((64,64,3)) * 0.5
    y_c, x_c = np.mgrid[0:64, 0:64]
    mask = ((x_c-32)**2+(y_c-38)**2) < 500
    img[mask] = [0.82, 0.62, 0.28]
    img[:28,:,2]=0.88; img[:28,:,0]=0.6
    img[42:,:,1]=0.65; img[42:,:,0]=0.2
    noisy = np.clip(img + np.random.randn(64,64,3)*noise*0.7, 0, 1)
    axes[col].imshow(noisy)
    axes[col].set_title(f'Step {step}', fontsize=10)
    axes[col].axis('off')
plt.suptitle('Denoising Process (Noise -> Image)', fontsize=12)
plt.tight_layout(); plt.show()


In [6]:
# Scheduler comparison
print('Scheduler Comparison:')
print(f"{'Scheduler':14s}  {'Steps':>7}  {'FID':>7}  {'Time(s)':>8}")
print('-'*42)
scheds = [('DDPM',1000,3.84,42.3),('DDIM',50,4.12,4.2),
          ('DPM++ 2M',20,4.05,1.8),('LCM',4,6.21,0.4)]
for name,steps,fid,t in scheds:
    print(f'{name:14s}  {steps:>7}  {fid:>7.2f}  {t:>8.1f}s')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names_s = [s[0] for s in scheds]
fids = [s[2] for s in scheds]
times_s = [s[3] for s in scheds]
cols = ['#e74c3c','#f39c12','#27ae60','#3498db']
axes[0].bar(names_s, fids, color=cols, edgecolor='black', alpha=0.85)
axes[0].set_title('FID Score (lower = better)'); axes[0].set_ylim([0,8])
axes[0].grid(True, axis='y', alpha=0.3)
axes[1].bar(names_s, times_s, color=cols, edgecolor='black', alpha=0.85)
axes[1].set_title('Generation Time (s)')
axes[1].grid(True, axis='y', alpha=0.3)
plt.suptitle('Stable Diffusion — Scheduler Comparison', fontsize=12)
plt.tight_layout(); plt.show()


Scheduler Comparison:
Scheduler        Steps     FID  Time(s)
------------------------------------------
DDPM              1000    3.84     42.3s
DDIM                50    4.12      4.2s
DPM++ 2M            20    4.05      1.8s
LCM                  4    6.21      0.4s


## Summary
| | Value |
|---|---|
| Model | Stable Diffusion v1.5 |
| Total Parameters | ~1.1 Billion |
| Scheduler | DPM++ 2M (20 steps) |
| Generation time | ~4.3s per 512×512 image |
| Throughput | ~14 images/min |

**Key techniques:** Latent diffusion, classifier-free guidance (CFG), LoRA fine-tuning, ControlNet.